In [1]:
import sys
import os

sys.path.insert(0, os.path.abspath("/home/ElasticNotebook"))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [3]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/ibtesama/getting-started-with-a-movie-recommendation-system/src/annotated/checkpoints/post_cell_0.pickle

[[<elastic.core.graph.variable_snapshot.VariableSnapshot object at 0x7f9ffc159810>], [<elastic.core.graph.variable_snapshot.VariableSnapshot object at 0x7f9ffc1598d0>], [<elastic.core.graph.variable_snapshot.VariableSnapshot object at 0x7f9ffc159270>], [<elastic.core.graph.variable_snapshot.VariableSnapshot object at 0x7f9ffc159900>], [<elastic.core.graph.variable_snapshot.VariableSnapshot object at 0x7f9ffc1f3f70>], [<elastic.core.graph.variable_snapshot.VariableSnapshot object at 0x7f9ffc159c00>]]
trying: ['df1']


me:  1
trying: ['np']
me:  0
trying: ['df2']
me:  1
trying: ['time']
me:  0
trying: ['orig_output']
me:  2
trying: ['pd']
me:  0
Declaring variable np
Declaring variable time
Declaring variable pd
Declaring variable df1
Declaring variable df2
Declaring variable orig_output


In [4]:
%%RecordEvent
%%time
### cell 1 ###
import numpy as np

factor = 50
if "IREWR_LESS_REPLICATION" in os.environ and os.environ["IREWR_LESS_REPLICATION"] == "True":
    factor //= 3

# generate a tiled index to replicate the entire frame in block order
indices = np.arange(len(df1))
replicated_idx = np.tile(indices, factor)
# use GPU‐accelerated .take to gather rows in the correct order
df1 = df1.take(replicated_idx).reset_index(drop=True)

df1.info()

<class 'cudf.core.dataframe.DataFrame'>
RangeIndex: 240150 entries, 0 to 240149
Data columns (total 4 columns):
 #   Column    Non-Null Count   Dtype
---  ------    --------------   -----
 0   movie_id  240150 non-null  int64
 1   title     240150 non-null  object
 2   cast      240150 non-null  object
 3   crew      240150 non-null  object
dtypes: int64(1), object(3)
memory usage: 1.6+ GB
CPU times: user 37 ms, sys: 3.54 ms, total: 40.5 ms
Wall time: 54.9 ms


In [5]:
%Checkpoint /home/dias-benchmarks/notebooks/ibtesama/getting-started-with-a-movie-recommendation-system/src/rewritten/o4_mini_high/checkpoints/post_cell_1_try_1.pickle

migration speed (bps): 744053012.9476594
---------------------------
variables to migrate:
time 72
pd 72
replicated_idx 112
factor 28
df2 48
indices 38536
np 72
orig_output 16
df1 48
---------------------------
variables to recompute:
[]
---------------------------
cells to recompute:
[]
Checkpoint saved to: /home/dias-benchmarks/notebooks/ibtesama/getting-started-with-a-movie-recommendation-system/src/rewritten/o4_mini_high/checkpoints/post_cell_1_try_1.pickle


In [6]:
%PrintCellInfo opt_cell_exec_info

======= Cell 0 =======
Input variables []
Active variables ['df1']
Intermediate variables ['df2']
Future variables []
Modified dataframes
======= Cell 1 =======
Input variables ['df1']
Active variables []
Intermediate variables ['replicated_idx', 'df1', 'factor', 'indices']
Future variables []
Modified dataframes
Saved cell execution info to opt_cell_exec_info


In [7]:
with open(
    "/home/dias-benchmarks/notebooks/ibtesama/getting-started-with-a-movie-recommendation-system/src/opt_cell_exec_info_1_try_1.pkl",
    "wb",
) as f:
    pickle.dump(opt_cell_exec_info[1], f)

In [8]:
opt_output = Out.get(4)

In [9]:
df1_opt = df1
%LoadCheckpoint /home/dias-benchmarks/notebooks/ibtesama/getting-started-with-a-movie-recommendation-system/src/annotated/checkpoints/post_cell_1.pickle
assert compare_df(df1_opt, df1)

import numpy as np
from elastic.core.common.pandas import is_type_styler

is_orig_output_pd = isinstance(orig_output, (pd.Series, pd.DataFrame, pd.Index))
is_opt_output_pd = isinstance(opt_output, (pd.Series, pd.DataFrame, pd.Index))
is_orig_output_array = isinstance(
    orig_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray)
)
is_opt_output_array = isinstance(
    opt_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray)
)
is_orig_output_styler = is_type_styler(type(orig_output))
is_opt_output_styler = is_type_styler(type(opt_output))
if is_orig_output_styler and is_opt_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_orig_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_opt_output_styler:
    assert opt_output.to_html() == orig_output

if is_orig_output_pd and is_opt_output_pd:
    assert orig_output.equals(opt_output)
# TODO(jie): this is a hack.
elif (
    (is_orig_output_pd or is_opt_output_pd)
    and (is_orig_output_array or is_opt_output_array)
) or (is_orig_output_array and is_opt_output_array):
    assert list(orig_output) == list(opt_output)
else:
    assert orig_output == opt_output

[[<elastic.core.graph.variable_snapshot.VariableSnapshot object at 0x7f9ffa575420>], [<elastic.core.graph.variable_snapshot.VariableSnapshot object at 0x7f9ffa5754e0>], [<elastic.core.graph.variable_snapshot.VariableSnapshot object at 0x7f9ffa5757e0>], [<elastic.core.graph.variable_snapshot.VariableSnapshot object at 0x7f9ffa575990>], [<elastic.core.graph.variable_snapshot.VariableSnapshot object at 0x7f9ffa5754b0>], [<elastic.core.graph.variable_snapshot.VariableSnapshot object at 0x7f9ffa5753c0>], [<elastic.core.graph.variable_snapshot.VariableSnapshot object at 0x7f9ffa575810>]]
trying: ['df2']
me:  1
trying: ['factor']
me:  3
trying: ['time']
me:  0
trying: ['orig_output']
me:  2
trying: ['df1']


me:  3
trying: ['pd']
me:  0
trying: ['np']
me:  0


Declaring variable time
Declaring variable pd
Declaring variable np
Declaring variable df2
Declaring variable orig_output
Declaring variable factor
Declaring variable df1


ValueError: Content of df1 and df2 do not match